# Edgetape Whisper Transcription (Colab GPU)

Transcribe videos using Whisper on GPU for faster processing.

## Setup
1. Get your backend URL and secret from `.env`
2. Run all cells
3. Enter the job ID when prompted

In [ ]:
!pip install -q yt-dlp faster-whisper httpx

In [ ]:
#@title Configuration
import os

#@markdown Enter your backend URL and secret:
BACKEND_URL = "http://localhost:8000" #@param {type:"string"}
COLAB_SECRET = "" #@param {type:"string"}

#@markdown Or set as environment variables:
# BACKEND_URL = os.environ.get("EDGETAPE_BACKEND_URL", "http://localhost:8000")
# COLAB_SECRET = os.environ.get("EDGETAPE_COLAB_SECRET", "")

print(f"Backend: {BACKEND_URL}")
print(f"Secret configured: {bool(COLAB_SECRET)}")

In [ ]:
#@title Download and Transcribe
import json
import tempfile
from pathlib import Path
import httpx

#@markdown Enter the Job ID to process:
JOB_ID = "" #@param {type:"string"}

def get_job(job_id: str) -> dict:
    """Fetch job info from backend."""
    headers = {}
    if COLAB_SECRET:
        headers["Authorization"] = f"Bearer {COLAB_SECRET}"
    resp = httpx.get(f"{BACKEND_URL}/api/jobs/{job_id}", headers=headers)
    resp.raise_for_status()
    return resp.json()

def download_video(url: str, output_path: Path) -> Path:
    """Download video using yt-dlp."""
    import yt_dlp
    ydl_opts = {
        "outtmpl": str(output_path / "%(id)s.%(ext)s"),
        "format": "best[height<=720]",  # Limit quality for speed
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
        return Path(filename)

def transcribe_with_whisper(video_path: Path) -> list[dict]:
    """Transcribe video using faster-whisper on GPU."""
    from faster_whisper import WhisperModel
    
    print("Loading Whisper model (GPU)...")
    model = WhisperModel("large-v3-turbo", device="cuda", compute_type="float16")
    
    print(f"Transcribing {video_path.name}...")
    segments, info = model.transcribe(
        str(video_path),
        vad_filter=True,
        language="es",
    )
    
    result = []
    for seg in segments:
        text = seg.text.strip()
        if text:
            result.append({
                "start": seg.start,
                "end": seg.end,
                "text": text,
            })
    
    print(f"Transcribed {len(result)} segments")
    return result

def send_transcription(job_id: str, segments: list[dict]):
    """Send transcription back to backend."""
    headers = {"Content-Type": "application/json"}
    if COLAB_SECRET:
        headers["Authorization"] = f"Bearer {COLAB_SECRET}"
    resp = httpx.post(
        f"{BACKEND_URL}/api/jobs/{job_id}/transcription",
        json={"segments": segments},
        headers=headers,
        timeout=120.0,
    )
    resp.raise_for_status()
    return resp.json()

# Main flow
if not JOB_ID:
    print("Please enter a Job ID above")
else:
    print(f"Processing job: {JOB_ID}")
    
    # Get job info
    job = get_job(JOB_ID)
    print(f"Job status: {job['status']}")
    print(f"Source URL: {job.get('source_url', 'N/A')}")
    
    if not job.get("source_url"):
        print("Error: Job has no source URL. Only YouTube jobs can be processed from Colab.")
    else:
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)
            
            # Download
            print("\nDownloading video...")
            video_path = download_video(job["source_url"], tmpdir)
            print(f"Downloaded: {video_path.name}")
            
            # Transcribe
            print("\nTranscribing...")
            segments = transcribe_with_whisper(video_path)
            
            # Send to backend
            print("\nSending transcription to backend...")
            result = send_transcription(JOB_ID, segments)
            print(f"Done! Job status: {result['status']}")
            print(f"Clips generated: {result.get('clip_count', 0)}")